In [1]:
# ============================================================
# ЛАБОРАТОРНАЯ РАБОТА №1
# Классификация изображений с использованием EfficientNet
# Трансферное обучение и тонкая настройка на CIFAR-10
# ============================================================

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, callbacks
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

# ============================================================
# КОНФИГУРАЦИЯ
# ============================================================

# Быстрый тест (уменьшите до True для проверки работоспособности)
QUICK_TEST = False  # Установите True для быстрого теста (5 эпох)

# Параметры обучения
BATCH_SIZE = 32
IMG_SIZE = 128
NUM_CLASSES = 10
EPOCHS_FE = 5 if QUICK_TEST else 30   # Feature Extraction
EPOCHS_FT = 5 if QUICK_TEST else 20   # Fine-Tuning

# Learning Rates
LR_FE = 0.0003   # Feature Extraction
LR_FT = 0.00001  # Fine-Tuning

# Разделение данных
TRAIN_SIZE = 8000
VAL_SIZE = 2000
TEST_SIZE = 2000

# Пути для сохранения результатов
OUTPUT_DIR = "lab1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Фиксация seed для воспроизводимости
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Имена классов CIFAR-10
CLASS_NAMES = [
    "Самолет", "Автомобиль", "Птица", "Кошка", "Олень",
    "Собака", "Лягушка", "Лошадь", "Корабль", "Грузовик"
]
CLASS_EMOJIS = ["✈️", "🚗", "🐦", "🐱", "🦌", "🐶", "🐸", "🐴", "🚢", "🚚"]

print("🔧 ЛАБОРАТОРНАЯ РАБОТА №1: EfficientNet на CIFAR-10")
print(f"📁 Результаты будут сохранены в: {OUTPUT_DIR}/")
print(f"🖥️  Устройство: {'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'}")

# ============================================================
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ============================================================

print("\n" + "=" * 60)
print("📊 ЭТАП 1: Загрузка и подготовка данных CIFAR-10")
print("=" * 60)

# Загрузка CIFAR-10
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

print(f"✅ Обучающая выборка (полная): {x_train_full.shape}")
print(f"✅ Тестовая выборка: {x_test.shape}")

# Разделение на train/val/test
x_train = x_train_full[:TRAIN_SIZE]
y_train = y_train_full[:TRAIN_SIZE]
x_val = x_train_full[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]
y_val = y_train_full[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]

# Нормализация к [0, 1]
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encoding для меток
y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_val_oh = to_categorical(y_val, NUM_CLASSES)
y_test_oh = to_categorical(y_test, NUM_CLASSES)

print(f"📦 Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

# ============================================================
# 2. АУГМЕНТАЦИЯ ДАННЫХ
# ============================================================

print("\n" + "=" * 60)
print("🔄 ЭТАП 2: Аугментация данных")
print("=" * 60)

# Генератор для обучающей выборки с аугментацией
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode="nearest",
)

# Генератор для val/test (только препроцессинг EfficientNet, без аугментации)
val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

# Функция специфичного препроцессинга EfficientNet
def preprocess_efficientnet(images):
    """Применяет препроцессинг EfficientNet к батчу изображений."""
    return applications.efficientnet.preprocess_input(images * 255.0)

# Применение препроцессинга EfficientNet
x_train_prep = preprocess_efficientnet(x_train)
x_val_prep = preprocess_efficientnet(x_val)
x_test_prep = preprocess_efficientnet(x_test)

# Ресайз до 128x128 (EfficientNet принимает разные размеры, но 128 - хороший баланс)
x_train_prep = tf.image.resize(x_train_prep, [IMG_SIZE, IMG_SIZE]).numpy()
x_val_prep = tf.image.resize(x_val_prep, [IMG_SIZE, IMG_SIZE]).numpy()
x_test_prep = tf.image.resize(x_test_prep, [IMG_SIZE, IMG_SIZE]).numpy()

print(f"✅ Препроцессинг EfficientNet применен")
print(f"✅ Размер изображений: {IMG_SIZE}×{IMG_SIZE}")

# ============================================================
# 3. ПОСТРОЕНИЕ МОДЕЛИ (FEATURE EXTRACTION)
# ============================================================

print("\n" + "=" * 60)
print("🏗️  ЭТАП 3: Построение модели EfficientNetB0")
print("=" * 60)

def build_model(trainable_base=False):
    """Создает модель EfficientNetB0 с классификационной головой."""
    # Базовая модель EfficientNetB0 (предобучена на ImageNet)
    base_model = applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling=None,
    )
    base_model.trainable = trainable_base

    # Классификационная голова
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model, base_model

# Построение модели для Feature Extraction
model, base_model = build_model(trainable_base=False)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print(f"✅ Модель построена")
print(f"📊 Всего параметров: {model.count_params():,}")
print(f"🔒 База EfficientNet: ЗАМОРОЖЕНА (Feature Extraction)")

model.summary()

# ============================================================
# 4. ОБУЧЕНИЕ ФАЗА 1: FEATURE EXTRACTION
# ============================================================

print("\n" + "=" * 60)
print(f"🎯 ЭТАП 4: Feature Extraction ({EPOCHS_FE} эпох)")
print("=" * 60)

# Callbacks для Feature Extraction
callbacks_fe = [
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

start_time = time.time()

history_fe = model.fit(
    x_train_prep, y_train_oh,
    validation_data=(x_val_prep, y_val_oh),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_FE,
    callbacks=callbacks_fe,
    verbose=1,
)

time_fe = time.time() - start_time
print(f"\n⏱️  Время Feature Extraction: {time_fe:.1f} секунд ({time_fe/60:.1f} мин)")

# ============================================================
# 5. ОБУЧЕНИЕ ФАЗА 2: FINE-TUNING
# ============================================================

print("\n" + "=" * 60)
print(f"🔥 ЭТАП 5: Fine-Tuning ({EPOCHS_FT} эпох)")
print("=" * 60)

# Размораживаем верхние слои базовой модели (последние 30 слоев)
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"🔓 Разморожено слоев: {sum(1 for l in base_model.layers if l.trainable)}")

# Перекомпиляция с новым, более низким learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FT),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Callbacks для Fine-Tuning
callbacks_ft = [
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

start_time = time.time()

history_ft = model.fit(
    x_train_prep, y_train_oh,
    validation_data=(x_val_prep, y_val_oh),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_FT,
    callbacks=callbacks_ft,
    verbose=1,
)

time_ft = time.time() - start_time
print(f"\n⏱️  Время Fine-Tuning: {time_ft:.1f} секунд ({time_ft/60:.1f} мин)")

# ============================================================
# 6. ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ
# ============================================================

print("\n" + "=" * 60)
print("📈 ЭТАП 6: Оценка на тестовой выборке")
print("=" * 60)

# Предсказания
y_pred_proba = model.predict(x_test_prep, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = y_test.flatten()

# Метрики
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
test_loss, test_acc = model.evaluate(x_test_prep, y_test_oh, verbose=0)

print(f"\n🏆 РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ:")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Loss: {test_loss:.4f}")
print(f"   • Macro F1-Score: {macro_f1:.4f}")

# Classification Report
print("\n📋 CLASSIFICATION REPORT:")
report = classification_report(
    y_true, y_pred,
    target_names=[f"{e} {n}" for e, n in zip(CLASS_EMOJIS, CLASS_NAMES)],
    digits=4,
)
print(report)

# ============================================================
# 7. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ============================================================

print("\n" + "=" * 60)
print("📊 ЭТАП 7: Визуализация результатов")
print("=" * 60)

# --- 7.1 Кривые обучения ---
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Объединяем истории обеих фаз
all_epochs = list(range(len(history_fe.history['loss']))) + \
             list(range(len(history_fe.history['loss']), 
                       len(history_fe.history['loss']) + len(history_ft.history['loss'])))

all_loss = history_fe.history['loss'] + history_ft.history['loss']
all_val_loss = history_fe.history['val_loss'] + history_ft.history['val_loss']
all_acc = history_fe.history['accuracy'] + history_ft.history['accuracy']
all_val_acc = history_fe.history['val_accuracy'] + history_ft.history['val_accuracy']

# Loss
axes[0, 0].plot(all_loss, label='Train Loss', linewidth=2)
axes[0, 0].plot(all_val_loss, label='Val Loss', linewidth=2)
axes[0, 0].axvline(x=len(history_fe.history['loss']), color='r', linestyle='--', 
                   label='Fine-Tuning start')
axes[0, 0].set_title('Потери во время обучения', fontsize=13)
axes[0, 0].set_xlabel('Эпохи')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(all_acc, label='Train Accuracy', linewidth=2)
axes[0, 1].plot(all_val_acc, label='Val Accuracy', linewidth=2)
axes[0, 1].axvline(x=len(history_fe.history['accuracy']), color='r', linestyle='--',
                   label='Fine-Tuning start')
axes[0, 1].set_title('Точность во время обучения', fontsize=13)
axes[0, 1].set_xlabel('Эпохи')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# --- 7.2 Матрица ошибок ---
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[f"{e}\n{n}" for e, n in zip(CLASS_EMOJIS, CLASS_NAMES)],
    yticklabels=[f"{e}\n{n}" for e, n in zip(CLASS_EMOJIS, CLASS_NAMES)],
    ax=axes[1, 0],
)
axes[1, 0].set_title("Матрица ошибок", fontsize=13)
axes[1, 0].set_xlabel("Предсказано")
axes[1, 0].set_ylabel("Истинно")

# --- 7.3 F1-Score по классам ---
from sklearn.metrics import f1_score as f1_func
f1_per_class = f1_func(y_true, y_pred, average=None)
colors = sns.color_palette("viridis", NUM_CLASSES)
axes[1, 1].barh([f"{e} {n}" for e, n in zip(CLASS_EMOJIS, CLASS_NAMES)], 
                f1_per_class, color=colors)
axes[1, 1].set_title("F1-Score по классам", fontsize=13)
axes[1, 1].set_xlabel("F1-Score")
axes[1, 1].set_xlim(0, 1)
axes[1, 1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(f1_per_class):
    axes[1, 1].text(v + 0.01, i, f"{v:.3f}", va='center', fontsize=9)

# Убираем пустой subplot
axes[0, 0].set_position([0.08, 0.55, 0.4, 0.38])
axes[0, 1].set_position([0.55, 0.55, 0.4, 0.38])
axes[1, 0].set_position([0.08, 0.08, 0.4, 0.38])
axes[1, 1].set_position([0.55, 0.08, 0.4, 0.38])

plt.suptitle(f"EfficientNetB0 на CIFAR-10 | Accuracy: {accuracy:.4f} | F1: {macro_f1:.4f}",
             fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_efficientnet_ft.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Матрица ошибок сохранена")

# --- 7.4 Примеры предсказаний ---
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
indices = np.random.choice(len(x_test), 15, replace=False)

for i, idx in enumerate(indices):
    ax = axes[i // 5, i % 5]
    # Показываем оригинальное изображение (до препроцессинга)
    img = x_test[idx]
    ax.imshow(img)
    
    pred_class = y_pred[idx]
    true_class = y_true[idx]
    prob = y_pred_proba[idx][pred_class] * 100
    
    color = "green" if pred_class == true_class else "red"
    title = f"{CLASS_EMOJIS[pred_class]} {CLASS_NAMES[pred_class]}\n{prob:.1f}%"
    ax.set_title(title, color=color, fontsize=10, fontweight='bold')
    ax.axis("off")

plt.suptitle("Примеры предсказаний EfficientNetB0 (Fine-Tuning)",
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUTPUT_DIR, "predictions_examples.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Примеры предсказаний сохранены")

# ============================================================
# 8. СОХРАНЕНИЕ МОДЕЛИ И ОТЧЕТА
# ============================================================

print("\n" + "=" * 60)
print("💾 ЭТАП 8: Сохранение модели и отчета")
print("=" * 60)

# Сохранение модели
model_path = os.path.join(OUTPUT_DIR, "efficientnet_cifar10.h5")
model.save(model_path)
print(f"✅ Модель сохранена: {model_path}")

# Текстовый отчет
report_path = os.path.join(OUTPUT_DIR, "lab1_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("ЛАБОРАТОРНАЯ РАБОТА №1 - ОТЧЕТ\n")
    f.write("EfficientNetB0 на CIFAR-10\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("КОНФИГУРАЦИЯ:\n")
    f.write("-" * 40 + "\n")
    f.write(f"• Размер изображений: {IMG_SIZE}×{IMG_SIZE}\n")
    f.write(f"• Batch size: {BATCH_SIZE}\n")
    f.write(f"• LR (Feature Extraction): {LR_FE}\n")
    f.write(f"• LR (Fine-Tuning): {LR_FT}\n")
    f.write(f"• Эпох FE: {len(history_fe.history['loss'])}\n")
    f.write(f"• Эпох FT: {len(history_ft.history['loss'])}\n\n")
    
    f.write("ВРЕМЕННЫЕ ХАРАКТЕРИСТИКИ:\n")
    f.write("-" * 40 + "\n")
    f.write(f"• Время Feature Extraction: {time_fe:.1f} с\n")
    f.write(f"• Время Fine-Tuning: {time_ft:.1f} с\n")
    f.write(f"• Всего параметров: {model.count_params():,}\n\n")
    
    f.write("РЕЗУЛЬТАТЫ НА ТЕСТЕ:\n")
    f.write("-" * 40 + "\n")
    f.write(f"• Accuracy: {accuracy:.4f}\n")
    f.write(f"• Loss: {test_loss:.4f}\n")
    f.write(f"• Macro F1-Score: {macro_f1:.4f}\n\n")
    
    f.write("CLASSIFICATION REPORT:\n")
    f.write("-" * 40 + "\n")
    f.write(report)

print(f"✅ Отчет сохранен: {report_path}")

# ============================================================
# ФИНАЛЬНАЯ СВОДКА
# ============================================================

print("\n" + "=" * 60)
print("🎉 ЛАБОРАТОРНАЯ РАБОТА №1 ЗАВЕРШЕНА!")
print("=" * 60)
print(f"📁 Все результаты в папке: {OUTPUT_DIR}/")
print(f"📊 Сгенерированные файлы:")
for file in os.listdir(OUTPUT_DIR):
    print(f"   • {file}")
print(f"\n🏆 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Macro F1: {macro_f1:.4f}")
print(f"   • Loss: {test_loss:.4f}")

c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


🔧 ЛАБОРАТОРНАЯ РАБОТА №1: EfficientNet на CIFAR-10
📁 Результаты будут сохранены в: lab1_results/
🖥️  Устройство: CPU

📊 ЭТАП 1: Загрузка и подготовка данных CIFAR-10
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 76s 0us/step


c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


✅ Обучающая выборка (полная): (50000, 32, 32, 3)
✅ Тестовая выборка: (10000, 32, 32, 3)
📦 Train: (8000, 32, 32, 3), Val: (2000, 32, 32, 3), Test: (10000, 32, 32, 3)

🔄 ЭТАП 2: Аугментация данных
✅ Препроцессинг EfficientNet применен
✅ Размер изображений: 128×128

🏗️  ЭТАП 3: Построение модели EfficientNetB0
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step
✅ Модель построена
📊 Всего параметров: 4,219,949
🔒 База EfficientNet: ЗАМОРОЖЕНА (Feature Extraction)


Model: "functional"

...


🎯 ЭТАП 4: Feature Extraction (30 эпох)
Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 435s 1s/step - accuracy: 0.6126 - loss: 1.2596 - val_accuracy: 0.8445 - val_loss: 0.4957 - learning_rate: 3.0000e-04
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 254s 1s/step - accuracy: 0.7933 - loss: 0.6584 - val_accuracy: 0.8615 - val_loss: 0.4288 - learning_rate: 3.0000e-04
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 231s 924ms/step - accuracy: 0.8248 - loss: 0.5491 - val_accuracy: 0.8655 - val_loss: 0.4172 - learning_rate: 3.0000e-04
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 240s 962ms/step - accuracy: 0.8465 - loss: 0.4611 - val_accuracy: 0.8670 - val_loss: 0.4088 - learning_rate: 3.0000e-04
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 238s 951ms/step - accuracy: 0.8606 - loss: 0.4140 - val_accuracy: 0.8670 - val_loss: 0.4023 - learning_rate: 3.0000e-04
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 247s 987ms/step - accuracy: 0.8764 - loss: 0.3743 - val_accuracy: 0.8720 - val_loss: 0.3994 - learning_rate: 3.0000e-04
Epoch 

c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\seaborn\utils.py:61: UserWarning: Glyph 128663 (\N{AUTOMOBILE}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\seaborn\utils.py:61: UserWarning: Glyph 128038 (\N{BIRD}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\seaborn\utils.py:61: UserWarning: Glyph 129420 (\N{DEER}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\seaborn\utils.py:61: UserWarning: Glyph 128054 (\N{DOG FACE}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\seaborn\utils.py:61: UserWarning: Glyph 128056 (\N{FROG FACE}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
c:\Users\Manuel_Oseias\anaconda3\Lib\site-packages\seaborn\utils.py:61: UserWarning: Glyph 128052 (\N{HORSE FACE}) missing from font(s) DejaVu Sans.
  fig.canvas.draw()
c

✅ Матрица ошибок сохранена


C:\Users\Manuel_Oseias\AppData\Local\Temp\ipykernel_9560\833766353.py:414: UserWarning: Glyph 128038 (\N{BIRD}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\Manuel_Oseias\AppData\Local\Temp\ipykernel_9560\833766353.py:414: UserWarning: Glyph 128663 (\N{AUTOMOBILE}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\Manuel_Oseias\AppData\Local\Temp\ipykernel_9560\833766353.py:414: UserWarning: Glyph 128674 (\N{SHIP}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\Manuel_Oseias\AppData\Local\Temp\ipykernel_9560\833766353.py:414: UserWarning: Glyph 128666 (\N{DELIVERY TRUCK}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\Manuel_Oseias\AppData\Local\Temp\ipykernel_9560\833766353.py:414: UserWarning: Glyph 128054 (\N{DOG FACE}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\Manuel_Oseias\AppData\Local\Temp\ipy

✅ Примеры предсказаний сохранены

💾 ЭТАП 8: Сохранение модели и отчета
✅ Модель сохранена: lab1_results\efficientnet_cifar10.h5
✅ Отчет сохранен: lab1_results\lab1_report.txt

🎉 ЛАБОРАТОРНАЯ РАБОТА №1 ЗАВЕРШЕНА!
📁 Все результаты в папке: lab1_results/
📊 Сгенерированные файлы:
   • confusion_matrix_efficientnet_ft.png
   • efficientnet_cifar10.h5
   • lab1_report.txt
   • predictions_examples.png

🏆 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:
   • Accuracy: 0.8737
   • Macro F1: 0.8732
   • Loss: 0.3925
